# Inference segmentation DGCNN

Notebook d'inference pour charger un checkpoint pre-entraine, choisir un scan de test, predire la segmentation sur le scan complet avec plusieurs vues si necessaire, puis afficher le resultat 3D.

Principe multi-vues: si le scan contient plus de points que l'entree utilisee par le checkpoint, le scan est decoupe en vues de `NUM_POINTS`. Chaque point recoit un ou plusieurs votes; son label final est le label predit le plus souvent. Les egalites sont departagees par la confiance moyenne.

## 1. Parametres d'execution

Checkpoint, split, scan et sampling multi-vues.

In [ ]:
from pathlib import Path

CHECKPOINT_PATH = Path("artifacts/augmentation/best.pt")
REQUESTED_SPLIT = "test"
DEVICE = "auto"

RANDOM_SEED = 42
SAMPLE_INDEX = None
EXCLUDED_TARGET_CLASSES = (1, 16)

NUM_POINTS = None
NUM_VIEW_PASSES = 10


## 2. Setup

In [ ]:
import random
import sys
from collections.abc import Iterator

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import BoundaryNorm, ListedColormap

try:
    from tqdm import tqdm
except Exception:
    def tqdm(iterable: Iterator, **_: object) -> Iterator:
        return iterable


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Impossible de trouver la racine du projet Orthotwin3D.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_segmentation import (
    build_model,
    configure_cuda_performance,
    validate_model_dataset_contract,
)
from src.datasets.teeth3ds_processed import Teeth3DSSegmentationDataset
from src.training.checkpointing import checkpoint_configs
from src.training.utils import get_device, set_seed


## 3. Fonctions utilitaires

In [ ]:
def resolve_checkpoint_path(path: str | Path) -> Path:
    checkpoint_path = Path(path).expanduser()
    if not checkpoint_path.is_absolute():
        checkpoint_path = PROJECT_ROOT / checkpoint_path
    if checkpoint_path.is_file():
        return checkpoint_path

    fallback_candidates = [
        PROJECT_ROOT / "artifacts" / "augmentation" / "best.pt",
        PROJECT_ROOT / "artifacts" / "baseline" / "best.pt",
        PROJECT_ROOT / "outputs" / "teethseg22_dgcnn_groupnorm" / "checkpoints" / "best.pt",
        PROJECT_ROOT / "outputs" / "teethseg22_dgcnn_groupnorm" / "checkpoints" / "last.pt",
    ]
    for candidate in fallback_candidates:
        if candidate.is_file():
            print(f"Checkpoint demande introuvable, utilisation de: {candidate}")
            return candidate
    raise FileNotFoundError(f"Aucun checkpoint trouve pour: {path}")


def split_has_samples(data_config: dict, split: str) -> bool:
    try:
        Teeth3DSSegmentationDataset.from_config(data_config, split=split, limit=1)
    except FileNotFoundError:
        return False
    return True


def choose_split(data_config: dict, requested_split: str) -> str:
    if split_has_samples(data_config, requested_split):
        return requested_split
    if requested_split != "val" and split_has_samples(data_config, "val"):
        print("Split 'test' absent dans les donnees locales; utilisation du split 'val'.")
        return "val"
    raise FileNotFoundError(f"Aucun sample disponible pour le split {requested_split!r} ou 'val'.")


def resolve_view_settings(train_config: dict, num_vertices: int) -> tuple[int, int, int, int]:
    sampling_config = train_config["sampling"]
    checkpoint_num_points = int(sampling_config["num_points"])
    checkpoint_view_passes = int(sampling_config["eval_views"])
    view_size = checkpoint_num_points if NUM_POINTS is None else int(NUM_POINTS)
    view_passes = checkpoint_view_passes if NUM_VIEW_PASSES is None else int(NUM_VIEW_PASSES)

    if view_size <= 0:
        raise ValueError("NUM_POINTS doit etre positif.")
    if view_passes <= 0:
        raise ValueError("NUM_VIEW_PASSES doit etre positif.")
    return min(view_size, num_vertices), view_passes, checkpoint_num_points, checkpoint_view_passes


def has_excluded_classes(sample: dict, excluded_classes: tuple[int, ...]) -> bool:
    if not excluded_classes:
        return False
    labels = sample["y"]
    excluded = torch.as_tensor(excluded_classes, dtype=labels.dtype)
    return bool(torch.isin(labels, excluded).any().item())


def select_sample_index(dataset: Teeth3DSSegmentationDataset, rng: random.Random) -> int:
    dataset_size = len(dataset)
    excluded_classes = tuple(int(label) for label in EXCLUDED_TARGET_CLASSES)
    if SAMPLE_INDEX is None:
        candidates = [
            index
            for index in range(dataset_size)
            if not has_excluded_classes(dataset[index], excluded_classes)
        ]
        if not candidates:
            raise ValueError(f"Aucun scan sans les classes exclues: {excluded_classes}")
        return rng.choice(candidates)
    index = int(SAMPLE_INDEX)
    if not 0 <= index < dataset_size:
        raise IndexError(f"SAMPLE_INDEX doit etre dans [0, {dataset_size}), recu {index}.")
    return index


def make_coverage_view_indices(
    num_vertices: int,
    view_size: int,
    view_passes: int,
    rng: random.Random,
) -> list[torch.Tensor]:
    if num_vertices <= 0:
        raise ValueError("Le scan ne contient aucun point.")
    if num_vertices <= view_size:
        return [torch.arange(num_vertices, dtype=torch.long)]

    generator = torch.Generator().manual_seed(rng.randrange(0, 2**31 - 1))
    all_indices = torch.arange(num_vertices, dtype=torch.long)
    views = []

    for _ in range(view_passes):
        permutation = all_indices[torch.randperm(num_vertices, generator=generator)]
        for start in range(0, num_vertices, view_size):
            view = permutation[start : start + view_size]
            if view.numel() < view_size:
                remaining = view_size - view.numel()
                mask = torch.ones(num_vertices, dtype=torch.bool)
                mask[view] = False
                pool = all_indices[mask]
                extra = pool[torch.randperm(pool.numel(), generator=generator)[:remaining]]
                view = torch.cat([view, extra])
            views.append(view.sort()[0])
    return views


def add_votes(
    votes: torch.Tensor,
    confidence_sums: torch.Tensor,
    sample_counts: torch.Tensor,
    indices: torch.Tensor,
    pred: torch.Tensor,
    confidence: torch.Tensor,
) -> None:
    votes.index_put_((indices, pred), torch.ones_like(pred, dtype=votes.dtype), accumulate=True)
    confidence_sums.index_put_((indices, pred), confidence.to(confidence_sums.dtype), accumulate=True)
    sample_counts.index_add_(0, indices, torch.ones_like(indices, dtype=sample_counts.dtype))


def majority_vote_predictions(votes: torch.Tensor, confidence_sums: torch.Tensor) -> torch.Tensor:
    max_votes = votes.max(dim=1, keepdim=True).values
    tied_labels = votes.eq(max_votes)
    tie_break_scores = confidence_sums.masked_fill(~tied_labels, -1.0)
    return tie_break_scores.argmax(dim=1)


def release_device_cache(device: torch.device) -> None:
    if device.type == "mps":
        torch.mps.empty_cache()
    elif device.type == "cuda":
        torch.cuda.empty_cache()


## 4. Charger le checkpoint et le modele

In [ ]:
checkpoint_path = resolve_checkpoint_path(CHECKPOINT_PATH)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
train_config, data_config = checkpoint_configs(checkpoint)

train_seed = int(train_config["run"]["seed"])
set_seed(train_seed)
selection_rng = random.Random(RANDOM_SEED)

device = get_device(DEVICE)
configure_cuda_performance(train_config.get("performance", {}), device)

split = choose_split(data_config, REQUESTED_SPLIT)
dataset = Teeth3DSSegmentationDataset.from_config(data_config, split=split)
model_config = train_config["model"]
validate_model_dataset_contract(model_config, dataset)

model = build_model(model_config).to(device)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

print(f"checkpoint: {checkpoint_path}")
print(f"epoch: {checkpoint.get('epoch')}")
print(f"device: {device}")
print(f"classes: {model_config['num_classes']} | input_channels: {model_config['input_channels']}")
print(f"checkpoint num_points: {train_config['sampling']['num_points']} | checkpoint eval_views: {train_config['sampling']['eval_views']}")
print(f"split utilise: {split} | scans disponibles: {len(dataset)}")


## 5. Choisir le scan et construire les vues

In [ ]:
sample_index = select_sample_index(dataset, selection_rng)
sample = dataset[sample_index]

scan_id = str(sample["scan_id"])
jaw = str(sample["jaw"])
num_vertices = int(sample["x"].shape[0])
view_size, view_passes, checkpoint_num_points, checkpoint_view_passes = resolve_view_settings(train_config, num_vertices)
view_indices = make_coverage_view_indices(num_vertices, view_size, view_passes, selection_rng)
target_classes = sorted(set(sample["y"].tolist()))

print(f"scan: {scan_id} | jaw: {jaw} | index: {sample_index}")
print(f"classes ground truth: {target_classes}")
print(f"points du scan: {num_vertices}")
print(f"taille entree modele utilisee: {view_size} (checkpoint: {checkpoint_num_points})")
print(f"passes: {view_passes} (checkpoint: {checkpoint_view_passes}) | vues totales: {len(view_indices)}")


## 6. Inference multi-vues avec vote majoritaire

In [ ]:
num_classes = int(model_config["num_classes"])
votes = torch.zeros((num_vertices, num_classes), dtype=torch.long)
confidence_sums = torch.zeros((num_vertices, num_classes), dtype=torch.float32)
sample_counts = torch.zeros(num_vertices, dtype=torch.long)

amp_enabled = bool(train_config["training"].get("amp", False) and device.type == "cuda")

for indices in tqdm(view_indices, desc="Inference views"):
    x_view = sample["x"].index_select(0, indices).unsqueeze(0).to(device)
    with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=amp_enabled):
        probas = model(x_view).softmax(dim=-1)

    view_pred = probas.argmax(dim=-1)[0].detach().cpu()
    view_confidence = probas.max(dim=-1).values[0].detach().cpu()
    add_votes(votes, confidence_sums, sample_counts, indices, view_pred, view_confidence)
    del x_view, probas

release_device_cache(device)

missing_points = int((sample_counts == 0).sum().item())
if missing_points:
    raise RuntimeError(f"{missing_points} points n'ont jamais ete echantillonnes.")

pred = majority_vote_predictions(votes, confidence_sums)
winning_votes = votes.gather(1, pred.unsqueeze(1)).squeeze(1)
confidence = confidence_sums.gather(1, pred.unsqueeze(1)).squeeze(1) / winning_votes.clamp_min(1).float()
target = sample["y"].detach().cpu()
points = sample["pos"].detach().cpu()


## 7. Resume des predictions

In [ ]:
point_accuracy = (pred == target).float().mean().item()
present_pred_classes = sorted(set(pred.tolist()))
present_target_classes = sorted(set(target.tolist()))

print(f"accuracy point-wise: {point_accuracy:.4f}")
print(f"confiance moyenne du label final: {confidence.mean().item():.4f}")
print(
    "votes par point: "
    f"min={sample_counts.min().item()} | "
    f"mean={sample_counts.float().mean().item():.2f} | "
    f"max={sample_counts.max().item()}"
)
print(f"classes ground truth presentes: {present_target_classes}")
print(f"classes predites presentes: {present_pred_classes}")


## 8. Visualisation 3D

In [ ]:
def segmentation_colormap(num_classes: int) -> tuple[ListedColormap, BoundaryNorm]:
    base = plt.get_cmap("tab20")
    colors = [(0.72, 0.72, 0.72, 1.0)] + [base(i) for i in range(1, num_classes)]
    cmap = ListedColormap(colors[:num_classes])
    norm = BoundaryNorm(np.arange(-0.5, num_classes + 0.5, 1), cmap.N)
    return cmap, norm


def set_axes_equal(ax, coords: np.ndarray) -> None:
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    ranges = np.maximum(maxs - mins, 1e-6)
    padding = ranges * 0.05
    ax.set_xlim(mins[0] - padding[0], maxs[0] + padding[0])
    ax.set_ylim(mins[1] - padding[1], maxs[1] + padding[1])
    ax.set_zlim(mins[2] - padding[2], maxs[2] + padding[2])
    ax.set_box_aspect(ranges)


def format_3d_axis(ax, coords: np.ndarray, title: str) -> None:
    ax.text2D(0.5, 0.88, title, transform=ax.transAxes, ha="center", va="top", fontsize=11, fontweight="bold")
    ax.set_axis_off()
    ax.set_facecolor("white")
    ax.view_init(elev=22, azim=-72)
    set_axes_equal(ax, coords)


def plot_segmentation(points: torch.Tensor, target: torch.Tensor, pred: torch.Tensor) -> plt.Figure:
    coords = points.numpy()
    target_np = target.numpy()
    pred_np = pred.numpy()
    cmap, norm = segmentation_colormap(int(model_config["num_classes"]))

    fig = plt.figure(figsize=(16, 4.8), constrained_layout=True, facecolor="white")
    ax_true = fig.add_subplot(1, 3, 1, projection="3d")
    ax_pred = fig.add_subplot(1, 3, 2, projection="3d")
    ax_error = fig.add_subplot(1, 3, 3, projection="3d")

    ax_true.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=target_np, cmap=cmap, norm=norm, s=0.9, linewidths=0, depthshade=False)
    ax_pred.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=pred_np, cmap=cmap, norm=norm, s=0.9, linewidths=0, depthshade=False)

    errors = pred_np != target_np
    correct = ~errors
    ax_error.scatter(coords[correct, 0], coords[correct, 1], coords[correct, 2], c="#d0d4d8", s=0.5, linewidths=0, alpha=0.18, depthshade=False)
    if errors.any():
        ax_error.scatter(coords[errors, 0], coords[errors, 1], coords[errors, 2], c="#d62828", s=1.4, linewidths=0, alpha=0.92, depthshade=False)

    format_3d_axis(ax_true, coords, "Ground truth")
    format_3d_axis(ax_pred, coords, "Prediction")
    format_3d_axis(ax_error, coords, "Errors")
    fig.suptitle(f"DGCNN tooth segmentation | accuracy {point_accuracy:.1%}", y=0.98, fontsize=13, fontweight="bold")
    return fig


fig = plot_segmentation(points, target, pred)
figure_path = PROJECT_ROOT / "doc" / "assets" / "inference_segmentation_baseline.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=240, bbox_inches="tight", facecolor="white")
print(f"figure sauvegardee: {figure_path}")
plt.show()
